# **Initialization**

In [1]:
print('Start')

Start


In [2]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import sys
import glob
import pulp
import vrplib
import re
import os
import gc
import contextlib
import modified_didppy as m_dp
import time as pytime



# **Data**

In [3]:
# Directory containing the VRP instances (Update this path)
# Note: Ensure this folder contains your .vrp files (e.g., A-n33-k5.vrp)
folder_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\3_CVRP_dual_bounds_and_models\Datasets\X"

# Get all .vrp files
all_files = glob.glob(os.path.join(folder_path, "*.vrp"))

# Select random instances (or all)
num_instances_to_test = 5
if len(all_files) > num_instances_to_test:
    selected_files = random.sample(all_files, num_instances_to_test)
else:
    selected_files = all_files

print(f"Found {len(all_files)} files. Selected {len(selected_files)} for testing.")
print("Selected Instances:")
for f in selected_files:
    print(f" - {os.path.basename(f)}")
print("-" * 50)

# ==========================================
# 2. Data Reading & Model Definition
# ==========================================

# Global variables to store current instance data (used by model creator)
current_num_locations = 0
current_num_vehicles = 0
current_capacity = 0
current_cust_demands = []
current_travel_cost = []

def read_formatted_data(file_path):
    """
    Reads a VRP file using vrplib and updates the global variables.
    """
    global current_num_locations, current_num_vehicles, current_capacity
    global current_cust_demands, current_travel_cost
    
    # Read instance
    instance = vrplib.read_instance(file_path)
    
    # Extract Capacity & Dimensions
    current_capacity = instance['capacity']
    current_num_locations = instance['dimension']
    
    # Extract Demands (Includes depot at index 0 with demand 0)
    current_cust_demands = instance['demand'].tolist()
    
    # Extract/Compute Edge Weights
    # vrplib usually computes euclidean distance automatically in 'edge_weight'
    current_travel_cost = instance['edge_weight'].tolist()
    
    # Try to determine number of vehicles from comment or filename
    # Defaulting to a safe upper bound (e.g., N) or a specific number if known
    # For Augerat instances (A-n33-k5), 'k5' means 5 trucks.
    import re
    match_comment = re.search(r"No of trucks:\s*(\d+)", str(instance.get('comment', '')))
    match_name = re.search(r"-k(\d+)", os.path.basename(file_path))
    
    if match_comment:
        current_num_vehicles = int(match_comment.group(1))
    elif match_name:
        current_num_vehicles = int(match_name.group(1))
    else:
        # Fallback: Estimate or set a high number (e.g., N)
        current_num_vehicles = current_num_locations 

    return current_num_locations, current_num_vehicles, current_travel_cost, current_capacity, current_cust_demands

Found 100 files. Selected 5 for testing.
Selected Instances:
 - X-n491-k59.vrp
 - X-n733-k159.vrp
 - X-n303-k21.vrp
 - X-n936-k151.vrp
 - X-n266-k58.vrp
--------------------------------------------------


# **1. Model and dual bound declaration**

In [4]:
def creation_of_didp_model_function():
    """
    Creates the CVRP DIDP model and returns it along with necessary metadata 
    for the heuristic functions.
    """
    # =========================================================
    # 1. Define Data
    # =========================================================
    n = current_num_locations
    m = current_num_vehicles
    q = current_capacity
    # Weights (demand)
    d = current_cust_demands

    # Distance matrix
    distance_list = current_travel_cost
    
    # =========================================================
    # 2. Define DIDP model
    # =========================================================
    model = m_dp.Model(float_cost= True)

    customer = model.add_object_type(number=n)
    unvisited_var = model.add_set_var(object_type=customer, target=list(range(1, n)), name='unvisited_customers')
    location_var = model.add_element_var(object_type=customer, target=0)
    load_var = model.add_float_resource_var(target=0, less_is_better=True)
    vehicles_var = model.add_int_resource_var(target=1, less_is_better=True)

    weight = model.add_float_table(d)
    distance_table = model.add_float_table(distance_list)

    model.add_base_case([unvisited_var.is_empty(), location_var == 0])

    for j in range(1, n):
        visit = m_dp.Transition(
            name=f"visit {j}",
            cost=distance_table[location_var, j] + m_dp.FloatExpr.state_cost(),
            effects=[
                (unvisited_var, unvisited_var.remove(j)),
                (location_var, j),
                (load_var, load_var + weight[j]),
            ],
            preconditions=[unvisited_var.contains(j), load_var + weight[j] <= q],
        )
        model.add_transition(visit)

    for j in range(1, n):
        visit_via_depot = m_dp.Transition(
            name=f"visit {j} with new vehicle",
            cost=distance_table[location_var, 0] + distance_table[0, j] + m_dp.FloatExpr.state_cost(),
            effects=[
                (unvisited_var, unvisited_var.remove(j)),
                (location_var, j),
                (load_var, weight[j]),
                (vehicles_var, vehicles_var + 1),
            ],
            preconditions=[unvisited_var.contains(j), vehicles_var < m],
        )
        model.add_transition(visit_via_depot)

    return_to_depot = m_dp.Transition(
        name="return",
        cost=distance_table[location_var, 0] + m_dp.FloatExpr.state_cost(),
        effects=[(location_var, 0)],
        preconditions=[unvisited_var.is_empty(), location_var != 0],
    )
    model.add_transition(return_to_depot)

    model.add_state_constr((m - vehicles_var + 1) * q - load_var >= weight[unvisited_var])

    # Min outgoing edge
    min_to = model.add_float_table(
        [min(distance_list[k][j] for k in range(n) if k != j) for j in range(n)]
    )
    model.add_dual_bound(min_to[unvisited_var] + (location_var != 0).if_then_else(min_to[0], 0))

    # Min incoming edge
    min_from = model.add_float_table(
        [min(distance_list[j][k] for k in range(n) if k != j) for j in range(n)]
    )
    model.add_dual_bound(
        min_from[unvisited_var] + (location_var != 0).if_then_else(min_from[location_var], 0)
    )
    # =========================================================
    # 3. Bundle Metadata
    # =========================================================
    metadata = {
        "unvisited_var": unvisited_var,
        "location_var": location_var,
        "distance_matrix": distance_list,
        "demand": d,
        "capacity": q,
        "num_vehicles": m,
        "num_nodes": n
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

# **Execution**

In [5]:
selected_files =['C:\\Users\\ACER\\Desktop\\Code\\0.Thesis implementation\\2_DIDP_custom_search_guidance_local\\Thesis_modified_DIDP\\3_CVRP_dual_bounds_and_models\\Datasets\\X\\X-n733-k159.vrp',
 'C:\\Users\\ACER\\Desktop\\Code\\0.Thesis implementation\\2_DIDP_custom_search_guidance_local\\Thesis_modified_DIDP\\3_CVRP_dual_bounds_and_models\\Datasets\\X\\X-n401-k29.vrp',
 'C:\\Users\\ACER\\Desktop\\Code\\0.Thesis implementation\\2_DIDP_custom_search_guidance_local\\Thesis_modified_DIDP\\3_CVRP_dual_bounds_and_models\\Datasets\\X\\X-n275-k28.vrp',
 'C:\\Users\\ACER\\Desktop\\Code\\0.Thesis implementation\\2_DIDP_custom_search_guidance_local\\Thesis_modified_DIDP\\3_CVRP_dual_bounds_and_models\\Datasets\\X\\X-n153-k22.vrp',
 'C:\\Users\\ACER\\Desktop\\Code\\0.Thesis implementation\\2_DIDP_custom_search_guidance_local\\Thesis_modified_DIDP\\3_CVRP_dual_bounds_and_models\\Datasets\\X\\X-n308-k13.vrp']

In [26]:
selected_files = ['C:\\Users\\ACER\\Desktop\\Code\\0.Thesis implementation\\2_DIDP_custom_search_guidance_local\\Thesis_modified_DIDP\\3_CVRP_dual_bounds_and_models\\Datasets\\X\\X-n733-k159.vrp',
 'C:\\Users\\ACER\\Desktop\\Code\\0.Thesis implementation\\2_DIDP_custom_search_guidance_local\\Thesis_modified_DIDP\\3_CVRP_dual_bounds_and_models\\Datasets\\X\\X-n153-k22.vrp']

In [6]:
results_data = []
output_csv_name = "CVRP_single_dual_bound_results.csv"

for i, file_path in enumerate(selected_files):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(selected_files)}] Processing: {instance_name}")
    
    try:
        # --- A. Read Data & Update Globals ---
        current_num_locations, current_num_vehicles, current_travel_cost, current_capacity, current_cust_demands = read_formatted_data(file_path)
        #current_capacity = current_capacity + 1000
        
        # --- B. Initialize Model ---
        model, state_data = creation_of_didp_model_function()
        
        # --- C. Solver Execution ---
        t_start = pytime.time()
        
        # Standard CABS solver with 30-minute limit
        solver = m_dp.CABS(
            model,
            quiet=False,
            time_limit= 10
        )
        
        solution = solver.search()
        
        t_end = pytime.time()
        duration = t_end - t_start
        
        # --- D. Logging Results ---
        if solution.is_optimal:
            cost = solution.cost
            status = "True"
        elif solution.cost is not None:
            cost = solution.cost
            status = "False (Time Limit)"
        else:
            cost = "Inf"
            status = "False (No Sol)"

        nodes_gen = solution.generated
        nodes_exp = solution.expanded
        print(f"Number of lcations: {current_num_locations}")
        print(f"Number of vehicles: {current_num_vehicles}")
        print(f"Total Capacity: {current_capacity*current_num_vehicles}")
        print(f"Total Demand: {sum(current_cust_demands)}")
        print(f"   -> Done. Cost: {cost}, Time: {duration:.2f}s, Optimal: {status}")

        results_data.append({
            "Instance": instance_name,
            "Cost": cost,
            "Nodes Expanded": nodes_exp,
            "Nodes Generated": nodes_gen,
            "Running Time (s)": duration,
            "Is Optimal": status,
            "Infeasibility": solution.is_infeasible
        })

    except Exception as e:
        print(f"   -> ERROR processing {instance_name}: {e}")
        # import traceback
        # traceback.print_exc()
        
        results_data.append({
            "Instance": instance_name,
            "Cost": "Error",
            "Nodes Expanded": 0,
            "Nodes Generated": 0,
            "Running Time (s)": 0,
            "Is Optimal": "Error"
        })

    # --- E. Intermediate Save ---
    df_results = pd.DataFrame(results_data)
    df_results.to_csv(output_csv_name, index=False)

print("\n" + "="*50)
print("Batch Testing Complete.")
print(f"Results saved to {output_csv_name}")
print(df_results)


[1/5] Processing: X-n733-k159.vrp
Number of lcations: 733
Number of vehicles: 159
Total Capacity: 3975
Total Demand: 3969
   -> Done. Cost: Inf, Time: 10.07s, Optimal: False (No Sol)

[2/5] Processing: X-n401-k29.vrp
Number of lcations: 401
Number of vehicles: 29
Total Capacity: 21605
Total Demand: 21275
   -> Done. Cost: 73419.87575827824, Time: 10.04s, Optimal: False (Time Limit)

[3/5] Processing: X-n275-k28.vrp
Number of lcations: 275
Number of vehicles: 28
Total Capacity: 280
Total Demand: 274
   -> Done. Cost: 24139.353189052774, Time: 10.02s, Optimal: False (Time Limit)

[4/5] Processing: X-n153-k22.vrp
Number of lcations: 153
Number of vehicles: 22
Total Capacity: 3168
Total Demand: 3068
   -> Done. Cost: Inf, Time: 10.00s, Optimal: False (No Sol)

[5/5] Processing: X-n308-k13.vrp
Number of lcations: 308
Number of vehicles: 13
Total Capacity: 3198
Total Demand: 3118
   -> Done. Cost: 33546.34949577914, Time: 10.02s, Optimal: False (Time Limit)

Batch Testing Complete.
Results 